In [1]:
pip install rembg opencv-python pillow numpy

Note: you may need to restart the kernel to use updated packages.


In [7]:
import cv2
import numpy as np
from rembg import remove
from PIL import Image

#inference function using trained YOLOv8 model
def detect_objects(image_path, model, target_class_name):
    results = model(image_path)
    detections = results[0].boxes.data.cpu().numpy()
    class_names = model.names

    #get detections of the target class only
    target_class_id = [i for i, name in class_names.items() if name == target_class_name][0]
    target_boxes = [box[:4] for box in detections if int(box[5]) == target_class_id]

    return target_boxes

def crop_and_remove_background(image_path, model, target_class_name, padding=20):
    #load original image
    image_bgr = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    # Get bounding boxes for the target class
    boxes = detect_objects(image_path, model, target_class_name)

    if not boxes:
        print(f"No objects of class '{target_class_name}' found.")
        return None

    results = []
    for i, (x1, y1, x2, y2) in enumerate(boxes):
        #add padding and crop to image size
        h, w = image_rgb.shape[:2]
        x1_p = max(int(x1) - padding, 0)
        y1_p = max(int(y1) - padding, 0)
        x2_p = min(int(x2) + padding, w)
        y2_p = min(int(y2) + padding, h)

        cropped = image_rgb[y1_p:y2_p, x1_p:x2_p]

        #convert cropped image to bytes for rembg
        cropped_pil = Image.fromarray(cropped)
        with BytesIO() as buffer:
            cropped_pil.save(buffer, format="PNG")
            input_bytes = buffer.getvalue()

        #remove background
        output_bytes = remove(input_bytes)
        result_image = Image.open(BytesIO(output_bytes)).convert("RGBA")
        results.append(result_image)

        #save to file
        result_image.save(f"result_{target_class_name}_{i}.png")

    return results  #list of PIL images

from ultralytics import YOLO
from io import BytesIO

model = YOLO("runs/train/product_model/weights/best.pt")  # e.g., 'runs/detect/train/weights/best.pt'
image_path = "images/train/IMG_4668.jpeg"
target_class = "product"

output_images = crop_and_remove_background(image_path, model, target_class)



image 1/1 /Users/andreanoh/Documents/Spring 2025 6010 ComputerVision/cv_product_info_extract/images/train/IMG_4668.jpeg: 640x480 1 quarter, 1 product, 87.3ms
Speed: 10.9ms preprocess, 87.3ms inference, 11.8ms postprocess per image at shape (1, 3, 640, 480)
